# Experiment 2.A: attack-rate-gradient symmetry check

Symmetry variant of Experiment 2. The 2D `deltas` bifurcation curves are continued from zero toward both positive and negative limits. The positive and negative continuations are saved separately before being combined into one centered plot.


In [ ]:
import os
from pathlib import Path

from AUTOclui import AUTOCommands as ac
from AUTOclui import runAUTO as ra
from pyvirtualdisplay import Display


In [ ]:
folder = Path.cwd()
os.chdir(folder)

model_name = 'common_model'
output_folder = folder / 'output_experiment_two_a_attack_rate_symmetry'
output_folder.mkdir(exist_ok=True)

parameter_index = 30
parameter_name = 'deltas'
parameter_limit = 0.4
mu_limits = [-0.25, 0.5]
continuation_step = 1.0e-3

parameter_file = folder / 'experiment_parameters.dat'
parameter_file.write_text('0.0 0.0\n')


In [ ]:
display = Display(visible=False, size=(1200, 900))
display.start()


In [ ]:
runner = ra.runAUTO()

def continue_from_zero(starting_point, direction, save_name):
    curve = ac.run(
        eq(starting_point),
        ICP=[28, parameter_index],
        ISW=2,
        DS=direction * continuation_step,
        DSMIN=1.0e-5,
        DSMAX=5.0e-3,
        NMX=8000,
        NPR=400,
        UZSTOP={28: mu_limits, parameter_index: [-parameter_limit, parameter_limit]},
        runner=runner,
    ).relabel()
    ac.save(curve, save_name)
    return curve

try:
    eq_forward = ac.run(e=model_name, c=model_name, runner=runner, NMX=4000, NPR=200)
    eq_backward = ac.run(DS='-', runner=runner, NMX=4000, NPR=200)
    eq = (eq_forward + eq_backward).relabel()
    ac.save(eq, 'eq')

    bp_curve_positive = continue_from_zero('BP1', 1.0, 'bp_curve_positive')
    bp_curve_negative = continue_from_zero('BP1', -1.0, 'bp_curve_negative')
    lp_curve_positive = continue_from_zero('LP1', 1.0, 'lp_curve_positive')
    lp_curve_negative = continue_from_zero('LP1', -1.0, 'lp_curve_negative')

    codim2 = (
        bp_curve_positive
        + bp_curve_negative
        + lp_curve_positive
        + lp_curve_negative
    ).relabel()
    ac.save(codim2, 'codim2')
finally:
    runner.config(clean=True)
    ac.clean()


In [ ]:
p = ac.plot('eq', hide=True)
p.config(
    stability=True,
    grid=False,
    bifurcation_x=['mu'],
    bifurcation_y=['PL'],
    xlabel='mu',
    ylabel='PL',
    title='',
    minx=0.0,
    maxx=0.1,
)
p.savefig(str(output_folder / 'experiment_two_a_attack_rate_symmetry_1d.png'))
p.savefig(str(output_folder / 'experiment_two_a_attack_rate_symmetry_1d.svg'))


In [ ]:
p = ac.plot('codim2', hide=True)
p.config(
    grid=False,
    bifurcation_x=[parameter_name],
    bifurcation_y=['mu'],
    xlabel=parameter_name,
    ylabel='mu',
    title='',
    minx=-parameter_limit,
    maxx=parameter_limit,
    miny=0.0,
    maxy=0.1,
)
p.savefig(str(output_folder / 'experiment_two_a_attack_rate_symmetry_2d.png'))
p.savefig(str(output_folder / 'experiment_two_a_attack_rate_symmetry_2d.svg'))


In [ ]:
display.stop()
parameter_file.unlink(missing_ok=True)
ac.delete('eq')
ac.delete('bp_curve_positive')
ac.delete('bp_curve_negative')
ac.delete('lp_curve_positive')
ac.delete('lp_curve_negative')
ac.delete('codim2')
